# Different Data Cleaning Methods in Data Engineering (Real-Time Use Cases)
...

Data cleaning is one of the most critical steps in data engineering pipelines. Raw data from different sources (APIs, databases, files, streaming systems) is usually inconsistent, incomplete, and noisy. Cleaning ensures the data is reliable for analytics, reporting, and machine learning.

---

# 1. Handling Missing Values (NULLs)

## Problem
Fields are empty or missing due to source system issues.

## Methods

### Remove records
```python
df.na.drop()
```

### Fill with default values
```python
df.na.fill(0)
df.na.fill("Unknown")
```

### Use COALESCE logic
```python
from pyspark.sql.functions import coalesce

df = df.withColumn("city", coalesce("city", "Unknown"))
```

## Real-Time Example
Customer data missing email or phone numbers.

---

# 2. Removing Duplicates

## Problem
Same record appears multiple times due to reprocessing or multiple sources.

## Methods
```python
df.distinct()
df.dropDuplicates(["id"])
```

## Real-Time Example
Duplicate orders from retry mechanisms in payment systems.

---

# 3. Standardizing Data Formats

## Problem
Different formats for same data.

Examples:
- Dates: `2025/01/01`, `01-01-2025`
- Case mismatch: `India`, `india`, `INDIA`

## Methods

### Standardize strings
```python
from pyspark.sql.functions import lower, trim

df = df.withColumn("country", lower(trim("country")))
```

### Standardize date format
```python
from pyspark.sql.functions import to_date

df = df.withColumn("date", to_date("date", "yyyy-MM-dd"))
```

## Real-Time Example
Multi-country customer datasets with inconsistent formats.

---

# 4. Removing Invalid Data

## Problem
Data that does not follow business rules.

## Methods

### Filter invalid values
```python
df = df.filter(df.age > 0)
```

### Range validation
```python
df = df.filter((df.salary > 1000) & (df.salary < 100000))
```

## Real-Time Example
Reject negative salary or invalid age values.

---

# 5. Handling Outliers

## Problem
Extreme values affecting analytics.

## Methods

### IQR method
```python
q1, q3 = df.approxQuantile("salary", [0.25, 0.75], 0.0)
iqr = q3 - q1

df = df.filter((df.salary >= q1 - 1.5*iqr) & (df.salary <= q3 + 1.5*iqr))
```

### Cap values (Winsorization)
```python
from pyspark.sql.functions import when

df = df.withColumn(
    "salary",
    when(df.salary > 100000, 100000).otherwise(df.salary)
)
```

## Real-Time Example
Salary or transaction amount spikes.

---

# 6. Data Type Correction

## Problem
Wrong or inconsistent data types.

## Methods

```python
df = df.withColumn("age", df.age.cast("int"))
df = df.withColumn("salary", df.salary.cast("double"))
```

## Real-Time Example
Numeric values stored as strings from CSV files.

---

# 7. Removing Special Characters

## Problem
Unwanted symbols in text fields.

## Methods

```python
from pyspark.sql.functions import regexp_replace

df = df.withColumn("name", regexp_replace("name", "[^a-zA-Z ]", ""))
```

## Real-Time Example
Cleaning customer names like `John@123`.

---

# 8. Trimming Whitespace

## Problem
Extra spaces affect joins and comparisons.

## Methods

```python
from pyspark.sql.functions import trim

df = df.withColumn("name", trim("name"))
```

## Real-Time Example
Join failure due to `"John "` vs `"John"`.

---

# 9. Standardizing Case

## Problem
Case mismatch causes duplicates logically.

## Methods

```python
from pyspark.sql.functions import lower

df = df.withColumn("email", lower("email"))
```

## Real-Time Example
Email deduplication.

---

# 10. Handling Inconsistent Data

## Problem
Multiple representations of same value.

Examples:
- "NY", "New York"
- "M", "Male"

## Methods

### Mapping dictionary
```python
from pyspark.sql.functions import when

df = df.withColumn(
    "gender",
    when(df.gender == "M", "Male")
    .when(df.gender == "F", "Female")
    .otherwise("Unknown")
)
```

## Real-Time Example
Standardizing gender or location values.

---

# 11. Removing Empty Strings

## Problem
Empty strings instead of NULL.

## Methods

```python
from pyspark.sql.functions import when, col

df = df.withColumn(
    "city",
    when(col("city") == "", None).otherwise(col("city"))
)
```

## Real-Time Example
Legacy systems storing blank values.

---

# 12. Handling Invalid Dates

## Problem
Wrong or corrupted dates.

## Methods

```python
df = df.withColumn("date", to_date("date"))
df = df.filter(df.date.isNotNull())
```

## Real-Time Example
Incorrect transaction timestamps.

---

# 13. Schema Validation

## Problem
Unexpected columns or missing fields.

## Methods

```python
expected_cols = ["id", "name", "age"]

df = df.select(*expected_cols)
```

## Real-Time Example
API payload changes.

---

# 14. Data Deduplication with Business Rules

## Problem
Duplicates exist but need business logic to resolve.

## Methods

```python
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.partitionBy("id").orderBy("updated_date")

df = df.withColumn("rn", row_number().over(window)).filter("rn=1")
```

## Real-Time Example
Keep latest customer profile.

---

# 15. Column Normalization

## Problem
Scale differences in numerical data.

## Methods

```python
from pyspark.sql.functions import col

df = df.withColumn("salary_norm", col("salary") / 100000)
```

## Real-Time Example
ML feature engineering.

---

# 16. Handling Corrupt Records

## Problem
Malformed CSV/JSON rows.

## Methods

```python
df = spark.read.option("mode", "PERMISSIVE").csv("/data")
```

### Modes
| Mode | Behavior |
|------|----------|
| PERMISSIVE | Keeps corrupt rows |
| DROPMALFORMED | Drops bad rows |
| FAILFAST | Fails pipeline |

## Real-Time Example
Streaming ingestion from unreliable sources.

---

# 17. Column Renaming and Standardization

## Problem
Inconsistent column names.

## Methods

```python
df = df.withColumnRenamed("EmpName", "employee_name")
```

## Real-Time Example
Multi-source ingestion (Oracle, APIs, CSVs).

---

# 18. Handling Join Issues (Data Cleaning Before Joins)

## Problem
Joins fail due to dirty keys.

## Methods

```python
df = df.withColumn("id", trim(col("id")))
```

## Real-Time Example
Fixing foreign key mismatches.

---

# Real-Time Data Cleaning Pipeline Example

```python
clean_df = (
    raw_df
    .dropDuplicates()
    .na.fill("Unknown")
    .withColumn("name", trim(lower("name")))
    .filter("age > 0")
    .withColumn("salary", col("salary").cast("double"))
)
```

---

# Real-Time Use Cases Summary

| Problem | Solution |
|----------|----------|
| Missing values | fill / drop / COALESCE |
| Duplicates | dropDuplicates / window functions |
| Invalid data | filter conditions |
| Inconsistent format | trim, lower, standardization |
| Outliers | IQR / capping |
| Wrong types | cast |
| Corrupt data | read modes |
| Dirty joins | cleaning keys |
| Schema mismatch | select expected columns |

---

# Interview Summary

## Q1. What is data cleaning?

Data cleaning is the process of correcting or removing inaccurate, incomplete, duplicate, or irrelevant data to improve data quality for downstream processing.

---

## Q2. What are most common cleaning steps in real-time pipelines?

- Handling NULLs
- Removing duplicates
- Standardizing formats
- Fixing data types
- Removing invalid records
- Handling outliers
- Normalizing data

---

## Q3. Where do you perform data cleaning in architecture?

- Bronze → Silver layer (most common in Lakehouse)
- ETL transformation layer
- Streaming preprocessing layer
- Data ingestion pipelines

---

# Best Practices

- ✅ Clean data as early as possible in the pipeline (Bronze → Silver).
- ✅ Always standardize schema and formats before joins.
- ✅ Maintain data quality rules (DQ checks).
- ✅ Avoid aggressive data loss unless business rules allow it.
- ✅ Log rejected or dirty records separately for auditing.
- ✅ Use deterministic rules for cleaning in production pipelines.